# Отчет по Контрольному Рубежу №1
**Студент:** Штыка Д.А
**Группа:** ИУ5-35Б
**Вариант:** 23

## Задание 1. Команды Linux
**Условие:**
Список всех выполняемых процессов переписать в файл Task_1. Создать в каталоге **sandbox** каталог **sandbox-2**, переместить туда файлы **Task_2, Task_3**. Переписать содержимое файла **/bin/ls** в файл **Task_2**, переименовав его в "**myls**", установить права доступа на файл "**myls**" - "**r-xr--r--**". Добавить в файл **Task_3** информацию о ядре системы.

In [ ]:
# Команды для выполнения задания:
ps aux > sandbox/Task_1
mkdir -p sandbox/sandbox-2
mv sandbox/Task_2 sandbox/Task_3 sandbox/sandbox-2/
cp /bin/ls sandbox/sandbox-2/myls
chmod 544 sandbox/sandbox-2/myls
uname -a >> sandbox/Task_3

**Результат выполнения:**

[МЕСТО ДЛЯ СКРИНШОТА 1: Выполнение команд и проверка ls -al sandbox/sandbox-2]

## Задание 2. Вычислительная задача
**Условие:**
Разработать программу на языке ассемблера, которая принимает в качестве параметра командной строки значение $n$, выполняет расчет указанного выражения и выводит результат на экран:
$$\sum_{k=1}^n k\cdot first\_digit(k) $$
где $first\_digit(k)$ - первая значащая цифра числа $k$.

In [ ]:
# Код программы (RK1/task2/task2.asm)
format ELF64
public _start

section '.data' writable
    res_buf db 32
    newline db 0x0A

section '.text' executable
_start:
    mov rsi, [rsp + 8]
    cmp rsi, 0
    je exit
    mov rdi, [rsi]
    mov rbx, 0

parse_n:
    cmp rdi, 0
    je start_calc
    mov al, [rdi]
    cmp al, '0'
    jl start_calc
    cmp al, '9'
    jg start_calc
    sub al, '0'
    imul rbx, 10
    add rbx, rax
    inc rdi
    jmp parse_n

start_calc:
    mov r12, 0
    mov r13, 1

loop_sum:
    cmp r13, rbx
    jg print_res
    mov rax, r13
    mov rdx, 0
find_first:
    cmp rax, 0
    je next_k
    mov rdx, rax
    mov rax, 10
    div rax
    cmp rax, 0
    jne find_first
    mov rax, rdx
    imul rax, r13
    add r12, rax
next_k:
    inc r13
    jmp loop_sum

print_res:
    mov rax, r12
    mov rdi, res_buf + 31
    mov rcx, 0
conv_num:
    cmp rax, 0
    je do_print
    mov rdx, 0
    mov rbx, 10
    div rbx
    add dl, '0'
    dec rdi
    mov [rdi], dl
    inc rcx
    jmp conv_num
do_print:
    mov rax, 1
    mov rdi, 1
    mov rsi, res_buf
    add rsi, 31
    sub rsi, rcx
    mov rdx, rcx
    syscall
    mov rax, 1
    mov rdi, 1
    mov rsi, newline
    mov rdx, 1
    syscall

exit:
    mov rax, 60
    xor rdi, rdi
    syscall

**Результат выполнения:**

[МЕСТО ДЛЯ СКРИНШОТА 2: Компиляция и запуск task2.out 10]

## Задание 3. Работа с файлами
**Условие:**
Написать программу на языке ассемблера, которая принимает два параметра командной строки: первый - имя некоторого каталога name, второй - некоторое число $n$. Программа заходит в данный каталог и создает в нем новый каталог с именем name, далее заходит в каталог `./name/name` и создает там новый каталог name, делая так $n$ раз.

In [ ]:
# Код программы (RK1/task3/task3.asm)
format ELF64
public _start

section '.data' writable

section '.text' executable
_start:
    mov rsi, [rsp + 8]
    cmp rsi, 0
    je exit
    
    mov rdi, [rsi]
    mov rsi, [rsi + 8]
    cmp rsi, 0
    je exit
    
    mov r12, [rsi]
    mov rsi, [rsi + 8]
    
    mov rbx, 0
parse_n:
    cmp rsi, 0
    je start_mkdir
    mov al, [rsi]
    cmp al, '0'
    jl start_mkdir
    cmp al, '9'
    jg start_mkdir
    sub al, '0'
    imul rbx, 10
    add rbx, rax
    inc rsi
    jmp parse_n

start_mkdir:
    mov r13, 0
loop_mkdir:
    cmp r13, rbx
    jge exit
    
    mov rax, 8
    mov rdi, r12
    mov rsi, 0777
    syscall
    
    mov rax, 12
    mov rdi, r12
    syscall
    
    inc r13
    jmp loop_mkdir

exit:
    mov rax, 60
    xor rdi, rdi
    syscall

**Результат выполнения:**

[МЕСТО ДЛЯ СКРИНШОТА 3: Компиляция и запуск task3.out mydir 3 и вывод ls -R]